In [1]:
from transformers import BigBirdTokenizer
import pandas as pd
import os
from tqdm import tqdm
import json

/home/mario/miniforge3/envs/xtemp-nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_id = 'google/bigbird-roberta-base'
tokenizer = BigBirdTokenizer.from_pretrained(model_id, trust_remote_code=True)

In [4]:
def convert_to_csv(input_path, output_path, tokenizer):
    assert os.path.exists(input_path)

    if not os.path.exists(output_path):
        os.makedirs(output_path)

    all_tokens_da = 0
    for train_or_dev in tqdm(os.listdir(input_path), 'Conversion to csv...'):
        train_or_dev_path = os.path.join(input_path, train_or_dev)

        df = {'entries': []}

        with open(train_or_dev_path, 'r') as f:
            objects = f.read().strip().split('\n')
            entries = [json.loads(obj) for obj in objects]

            raw_corpus = ""
            for entry in entries:
                if entry.get('exp_upd', None) is not None:
                    if entry['exp_upd']:
                        raw_corpus += entry['exp_upd'] + '\n\n'
                        df['entries'].append(entry['exp_upd'])
                    else:
                        raw_corpus += entry['exp'] + '\n\n'
                        df['entries'].append(entry['exp'])
                else:
                    continue

            raw_corpus = raw_corpus.strip()
            tokens_corpus = tokenizer.tokenize(raw_corpus)

            print(f'{train_or_dev} has {round(len(tokens_corpus) / (10 ** 6), 4)}M tokens for domain adaptation')

            all_tokens_da += len(tokens_corpus)

        df = pd.DataFrame(df)

        savename = 'dev.csv' if 'dev' in train_or_dev else 'train.csv'

        df.to_csv(os.path.join(output_path, savename), index=False)

    print(f'Total of {round(all_tokens_da / (10 ** 9), 4)}B tokens for domain adaptation')


convert_to_csv('raw_baseline_data', 'baseline_data', tokenizer)

Conversion to csv...:  50%|█████     | 1/2 [00:00<00:00,  3.44it/s]

dev_explanation.jsonl has 0.2418M tokens for domain adaptation
train_exp_unfinished_q_leaked.jsonl has 7.6398M tokens for domain adaptation


Conversion to csv...: 100%|██████████| 2/2 [00:09<00:00,  4.91s/it]

Total of 0.0079B tokens for domain adaptation


In [9]:
with open('raw_baseline_data/dev_explanation.jsonl', 'r') as f:
    lines = f.read().strip().split('\n')
    dev = [json.loads(line) for line in lines]

    test_data = []
    for test_example in tqdm(dev, desc='converting to test QA'):
        test_dict = {
            'question': test_example['question'],
            'cop': test_example['cop'],
            'opa': test_example['opa'],
            'opb': test_example['opb'],
            'opc': test_example['opc'],
            'opd': test_example['opd']
        }
        test_data.append(test_dict)

with open('baseline_data/test.json', 'w') as fw:
    json.dump(test_data, fw, indent=2)

converting to test QA: 100%|██████████| 4183/4183 [00:00<00:00, 1405493.36it/s]
